# Employee Attrition Prediction - Random Forest

This notebook uses an existing CSV dataset. It **does not generate the dataset**.

### Workflow
1. Upload `employee_attrition_dataset.csv`
2. Load and inspect the data
3. Preprocess salary
4. Train/test split
5. Train Random Forest
6. Evaluate Accuracy, Precision, Recall, F1-score and ROC-AUC
7. Display confusion matrix and feature importance
8. Enter employee values at runtime and predict whether the employee will leave or stay


In [ ]:
# STEP 1: Import libraries
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


In [ ]:
# STEP 2: Upload the existing dataset
uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Loaded file:", filename)
print("Dataset shape:", df.shape)
display(df.head(10))


In [ ]:
# STEP 3: Check dataset details
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nAttrition counts:")
print(df['left'].value_counts())

print("\nAttrition percentage:")
print((df['left'].value_counts(normalize=True) * 100).round(2))


In [ ]:
# STEP 4: Prepare features and target
X = df.drop(columns=['employee_id', 'left'])
y = df['left']

X = pd.get_dummies(
    X,
    columns=['salary'],
    drop_first=False
)

print("Features used for training:")
print(X.columns.tolist())


In [ ]:
# STEP 5: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


In [ ]:
# STEP 6: Train Random Forest model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully.")


In [ ]:
# STEP 7: Predict test data
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("First 10 predictions:", y_pred[:10])
print("First 10 probabilities:", np.round(y_prob[:10], 3))


In [ ]:
# STEP 8: Performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score : {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# STEP 9: Confusion matrix graph
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Employee Attrition - Confusion Matrix")
plt.show()


In [ ]:
# STEP 10: Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

display(feature_importance)

top_features = feature_importance.head(8)

plt.figure(figsize=(9, 5))
plt.barh(
    top_features['Feature'][::-1],
    top_features['Importance'][::-1]
)
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top Factors Driving Employee Attrition")
plt.tight_layout()
plt.show()


## Runtime Employee Input

Run the following cell whenever you want to enter a new employee's details.

The model will return:
- `0 = STAY`
- `1 = LEAVE`
- Probability of leaving


In [ ]:
# STEP 11: Runtime input prediction

print("Enter employee details below:\n")

satisfaction_level = float(input("Satisfaction level (0 to 1): "))
last_evaluation = float(input("Last evaluation score (0 to 1): "))
number_project = int(input("Number of projects: "))
average_monthly_hours = int(input("Average monthly working hours: "))
time_spend_company = int(input("Years spent in company: "))
work_accident = int(input("Work accident? (0 = No, 1 = Yes): "))
promotion_last_5years = int(input("Promotion in last 5 years? (0 = No, 1 = Yes): "))
salary = input("Salary bracket (low/medium/high): ").strip().lower()

new_employee = pd.DataFrame([{
    'satisfaction_level': satisfaction_level,
    'last_evaluation': last_evaluation,
    'number_project': number_project,
    'average_monthly_hours': average_monthly_hours,
    'time_spend_company': time_spend_company,
    'work_accident': work_accident,
    'promotion_last_5years': promotion_last_5years,
    'salary_low': salary == 'low',
    'salary_medium': salary == 'medium',
    'salary_high': salary == 'high'
}])

new_employee = new_employee.reindex(columns=X.columns, fill_value=0)

prediction = model.predict(new_employee)[0]
probability = model.predict_proba(new_employee)[0, 1]

print("\n------------------------------")
print("ATTRITION PREDICTION RESULT")
print("------------------------------")

if prediction == 1:
    print("Prediction: LEAVE (1)")
else:
    print("Prediction: STAY (0)")

print(f"Probability of leaving: {probability:.2%}")


### Example runtime input

```text
Satisfaction level: 0.30
Last evaluation score: 0.82
Number of projects: 6
Average monthly working hours: 255
Years spent in company: 5
Work accident: 0
Promotion in last 5 years: 0
Salary bracket: low
```

The output will be in this form:

```text
ATTRITION PREDICTION RESULT
Prediction: LEAVE (1)
Probability of leaving: XX.XX%
```
